# LightGBM／XGBoost＋特徵工程（2014 起）

- **核心原理**：lag、滾動統計、日期特徵轉成表格問題
- **變體**：`mlforecast`（Nixtla，自動做 lag／滾動特徵）、XGBoost、CatBoost；直接法（direct）vs 遞迴法（recursive）的多步預測
- **推薦資料集**：Store Sales（Kaggle「Store Sales - Time Series Forecasting」：厄瓜多 Favorita 超市，54 家店 × 33 個商品類別，2013-01～2017-08 每日銷售）
- **實作**：✋ 手刻特徵工程（`groupby` + `shift`／`rolling`）＋ 遞迴預測；🧰 LightGBM 當模型

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

和第 1～7 章的差別：

| | 第 1～7 章（統計模型） | 本章（機器學習） |
| --- | --- | --- |
| 序列數 | 一條（或幾條） | **1,782 條**（店 × 類別）一起 |
| 模型 | 每條序列一個模型（指數平滑、ARIMA…） | **一個全域模型**（global model）同時學所有序列 |
| 怎麼用時間 | 模型本身的遞迴／結構 | **把時間變成特徵**：過去的值（lag）、滾動平均、星期幾、發薪日、節日、促銷 → 一般的表格回歸 |
| 優點 | 可解釋、資料少也能用 | 序列之間共享資訊、容易加外生變數、Kaggle 時間序列比賽的主力 |
| 最大的陷阱 | — | **資料洩漏**：特徵不小心用到「預測當下還看不到的未來」 |

一句話：**把「預測 $y_{t+h}$」改寫成「用在時間 $t$ 已經知道的東西當特徵，回歸 $y_{t+h}$」—— 只要特徵做得對（沒有偷看未來），任何表格模型（第一部分的 GBDT）都可以拿來預測時間序列。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：`load_store_sales()` 回傳 Kaggle 的資料表。本章用到：

| 表 | 欄位 | 用途 |
| --- | --- | --- |
| `train` | `date`、`store_nbr`、`family`、`sales`、`onpromotion` | 每天每店每類的銷售額（目標）與促銷中的商品數 |
| `holidays_events` | `date`、`type`、`locale`、`description`、`transferred` | 節日（全國性的才用） |
| `oil` | `date`、`dcoilwtico` | 油價（厄瓜多的經濟跟石油有關；週末缺值） |

（如果 `load_store_sales()` 只回傳 `train` 的 DataFrame，本章會自動跳過節日與油價特徵。）

**切分**（跟 Kaggle 一樣預測未來 **16 天**）：test = 最後 16 天；validation = test 之前的 16 天；train = 更早的（從 2015-06-01 開始，夠算一年的 lag 就好，也比較快）。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $i$ | 序列（一家店的一個類別） | 店數 × 33 |
| $y_{i,t}$ | 銷售額 | ≥ 0，很多 0 |
| $H$ | 預測天數 | 16 |
| lag $L$ | $y_{i,t-L}$ | |
| 滾動平均 | $\frac1w\sum_{k=0}^{w-1}y_{i,t-L-k}$（從 lag $L$ 開始往前 $w$ 天） | |
| RMSLE | $\sqrt{\frac1n\sum(\log(1 + \hat y) - \log(1 + y))^2}$ | Kaggle 的官方指標 |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第一部分第 12 章 | GBDT（House Prices） | 一棵一棵的樹擬合殘差；LightGBM：leaf-wise、直方圖分箱、`num_leaves`、`learning_rate`、early stopping；類別特徵可以直接用。 |
| 第 1 章 | 指數平滑 | 季節性 naive（`y.shift(m)`）、滾動起點 CV、「只能用過去預測未來」。 |
| 第 7 章 | Prophet | 把時間展開成特徵（傅立葉、節日）再回歸；節日表的資訊洩漏（2016 年的季後賽日期在 2015 年還不知道）。本章的 lag 特徵也要注意同樣的事。 |
| 第 2 章 | ARIMA | AR = 「用自己的 lag 當特徵的線性回歸」；遞迴預測：把預測值回填當下一步的輸入。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. log1p 目標**：$z = \log(1 + y)$。銷售額跨好幾個數量級（書 2 元、雜貨 3,000 元）而且有很多 0 → 在 $z$ 上做回歸，預測再用 $\hat y = e^{\hat z} - 1$。在 $z$ 上最小化平方誤差 ≈ 直接最小化 RMSLE。

**2. 直接法 vs 遞迴法**（多步預測）：要預測 $t+1..t+16$：
- **直接法**：特徵只用「在 $t$ 時已知」的東西 → lag 至少 16（$y_{t+16-16} = y_t$）。一個模型預測全部 16 天，不用回填。
- **遞迴法**：模型用 lag 1 的特徵（預測明天），預測第 2 天時把第 1 天的「預測值」當 lag 1 填回去，依此類推 —— 誤差會累積。

**3. WAPE**：$\frac{\sum|y - \hat y|}{\sum y}$ —— 加權的平均絕對百分比誤差，大的序列權重大（業界常用）。

#### 程式

**4. `df.groupby(keys)["sales"].shift(L)`**：**在每條序列內部**往後移 $L$ 列（序列之間不會混在一起）。前提：資料依（序列, 日期）排好，而且每條序列的日期是連續的。直接 `df["sales"].shift(L)` 會把上一條序列的尾巴移到下一條的開頭 → 錯。

**5. `s.groupby(keys).transform(lambda x: x.rolling(w, min_periods=1).mean())`**：每條序列內部的滾動平均，結果跟原本的列對齊。

**6. `pd.Series.dt.day_of_week`、`.dt.day`、`.dt.is_month_end`**；`df.merge(other, on="date", how="left")`。

**7. LightGBM**：`lgb.LGBMRegressor(n_estimators, learning_rate, num_leaves, ...)`；`.fit(X, y, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100)])` → `best_iteration_`；`category` 型別的欄位會被當成類別特徵。

---
## 1. 資料與探索性資料分析（EDA）

> 執行時間提示：完整的 Kaggle 資料約 300 萬列。特徵工程約 1～2 分鐘、每次 LightGBM 訓練約 1～3 分鐘。想先快速跑一遍，可以把下一格的 `MAX_STORES` 設成 10（只用前 10 家店）。

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ml_dl_llm.data import load_store_sales

warnings.filterwarnings("ignore", category=FutureWarning)
MAX_STORES = None  # 例如 10：只用 store_nbr ≤ 10，跑得比較快
H = 16

store = load_store_sales()
raw = store["train"] if isinstance(store, dict) else store
holidays_raw = store.get("holidays_events") if isinstance(store, dict) else None
oil_raw = store.get("oil") if isinstance(store, dict) else None
df = raw[["date", "store_nbr", "family", "sales", "onpromotion"]].copy()
df["date"] = pd.to_datetime(df["date"])
if MAX_STORES is not None:
    df = df[df["store_nbr"] <= MAX_STORES]
KEYS = ["store_nbr", "family"]
all_days = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
missing_days = all_days.difference(df["date"].unique())
print("缺的日期：", [d.strftime("%Y-%m-%d") for d in missing_days])
# 把每條序列補齊每一天（缺的日期 = 店沒開：sales、onpromotion 補 0）→ 之後 groupby.shift(L) 就剛好是「L 天前」
grid = pd.MultiIndex.from_product([sorted(df["store_nbr"].unique()), sorted(df["family"].unique()), all_days], names=KEYS + ["date"])
df = df.set_index(KEYS + ["date"]).reindex(grid).fillna({"sales": 0.0, "onpromotion": 0}).reset_index()
df["onpromotion"] = df["onpromotion"].astype(int)
df = df.sort_values(KEYS + ["date"]).reset_index(drop=True)
n_series = df.groupby(KEYS).ngroups
print(f"補齊後 {len(df):,} 列；{df['store_nbr'].nunique()} 家店 × {df['family'].nunique()} 類 = {n_series} 條序列；{df['date'].min().date()} ～ {df['date'].max().date()}")
print(f"sales = 0 的比例：{(df['sales'] == 0).mean():.1%}；每條序列都 0 的序列數：{int((df.groupby(KEYS)['sales'].sum() == 0).sum())}")

In [ ]:
# 總銷售額、星期幾、日期（發薪日）、促銷
# 觀察：
#   - 總銷售額的趨勢與年季節？元旦那天怎麼了？
#   - 星期幾最高？每月幾號特別高（厄瓜多公部門 15 號與月底發薪）？
#   - 有促銷的列，銷售額比沒促銷的高多少？這是「促銷造成的」嗎？
daily = df.groupby("date")["sales"].sum()
fig, axes = plt.subplots(2, 2, figsize=(16, 7))
daily.plot(ax=axes[0, 0], lw=0.5)
daily.rolling(28).mean().plot(ax=axes[0, 0], color="tab:red")
axes[0, 0].set_title("total daily sales (28-day mean in red)")
sns.barplot(x=daily.index.day_of_week, y=daily.to_numpy(), ax=axes[0, 1], errorbar=None)
axes[0, 1].set_title("total sales by weekday (0 = Mon)")
sns.barplot(x=daily.index.day, y=daily.to_numpy() / daily.rolling(28, center=True, min_periods=1).mean().to_numpy(), ax=axes[1, 0], errorbar=None)
axes[1, 0].set_title("sales / 28-day centered mean, by day of month")
fam_sales = df.groupby("family")["sales"].sum().sort_values()
fam_sales.tail(15).plot.barh(ax=axes[1, 1])
axes[1, 1].set_title("total sales by family (top 15)")
plt.tight_layout()
plt.show()
print("元旦的總銷售：", daily[daily.index.strftime("%m-%d") == "01-01"].round(0).to_dict())
promo = df[df["sales"] > 0].assign(has_promo=lambda d: d["onpromotion"] > 0)
print("（sales > 0 的列）有促銷 vs 沒促銷的 log1p(sales) 平均：", promo.groupby("has_promo")["sales"].apply(lambda s: np.log1p(s).mean()).round(3).to_dict())

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 規模：有幾條序列？sales = 0 的比例多少？「整條序列都是 0」代表什麼（那家店根本不賣這一類）？這種序列要怎麼預測？
# (2) 缺的日期是哪些？為什麼（12/25 不開門）？本章把它們補成 sales = 0。元旦的銷售額也接近 0 —— 模型需要知道「今天是元旦」嗎？
# (3) 星期幾、發薪日（15 號、月底、1 號）的效果大嗎？
# (4) 促銷：有促銷的列平均銷售高很多。但「大的店、大的類別」本來就比較常促銷 —— 這個差距是因果還是混淆？模型裡同時有 store、family、lag 時，促銷特徵的效果會變大還是變小？
# (5) 類別：最大的幾類佔了多少？RMSLE 是在 log 尺度上算的 —— 書（很小）和雜貨（很大）的誤差在 RMSLE 裡的權重一樣嗎？
# 答：

### 切分

| | 日期 | 用途 |
| --- | --- | --- |
| train | 2015-06-01 ～ valid 開始前一天 | 訓練 |
| valid | test 之前的 16 天 | early stopping、選特徵、調參 |
| test | 最後 16 天 | 最後評估一次 |

特徵在**整份資料**上算（lag 需要更早的資料），但每一列的特徵只用到「那一列的日期往前 ≥ 16 天」的銷售（直接法）—— 所以 test 的特徵不會用到 test 期間的銷售。

In [ ]:
END = df["date"].max()
TEST_START = END - pd.Timedelta(days=H - 1)
VALID_START = TEST_START - pd.Timedelta(days=H)
TRAIN_START = pd.Timestamp("2015-06-01")
print("train：", TRAIN_START.date(), "～", (VALID_START - pd.Timedelta(days=1)).date(), "；valid：", VALID_START.date(), "～", (TEST_START - pd.Timedelta(days=1)).date(),
      "；test：", TEST_START.date(), "～", END.date())
is_tr = (df["date"] >= TRAIN_START) & (df["date"] < VALID_START)
is_va = (df["date"] >= VALID_START) & (df["date"] < TEST_START)
is_te = df["date"] >= TEST_START

---
## 2. Baseline（在 valid 上，只用 valid 開始前的資料）

| Baseline | 每條序列的預測 |
| --- | --- |
| 最後一天 | valid 前一天的銷售 |
| 最近 14 天平均 | |
| 季節性 naive（7 天） | 上週同一天（第 1 天～第 7 天用最後一週；第 8～16 天再重複一次） |
| 最近 4 週同一個星期幾的平均 | |

**指標**：RMSLE（主要）、MAE、WAPE。

In [ ]:
def scores(y: np.ndarray, pred: np.ndarray) -> dict:
    y, pred = np.asarray(y, dtype=float), np.clip(np.asarray(pred, dtype=float), 0, None)
    return {"rmsle": float(np.sqrt(np.mean((np.log1p(pred) - np.log1p(y)) ** 2))), "mae": float(np.mean(np.abs(y - pred))), "wape": float(np.abs(y - pred).sum() / y.sum())}


def baseline_predict(df: pd.DataFrame, start: pd.Timestamp, kind: str) -> np.ndarray:
    """df 的 [start, start + H) 期間每一列的 baseline 預測（只用 start 之前的資料）。"""
    hist = df[df["date"] < start]
    target = df[(df["date"] >= start) & (df["date"] < start + pd.Timedelta(days=H))]
    wide = hist.pivot_table(index=KEYS, columns="date", values="sales").iloc[:, -28:]
    wide = wide.T.reindex(pd.date_range(start - pd.Timedelta(days=28), start - pd.Timedelta(days=1))).ffill().T  # 補 12/25 這種缺的日期
    if kind == "last":
        val = wide.iloc[:, -1]
        return target.join(val.rename("p"), on=KEYS)["p"].to_numpy()
    if kind == "mean14":
        return target.join(wide.iloc[:, -14:].mean(axis=1).rename("p"), on=KEYS)["p"].to_numpy()
    last_week = wide.iloc[:, -7:]
    dow_last = {d.day_of_week: c for c, d in enumerate(last_week.columns)}
    if kind == "snaive7":
        col = target["date"].dt.day_of_week.map(dow_last).to_numpy()
        arr = last_week.reindex(pd.MultiIndex.from_frame(target[KEYS])).to_numpy()
        return arr[np.arange(len(target)), col]
    if kind == "dow4":
        out = np.zeros(len(target))
        last28 = wide.iloc[:, -28:]
        for dow in range(7):
            cols = [c for c in last28.columns if c.day_of_week == dow]
            m = last28[cols].mean(axis=1)
            rows = target["date"].dt.day_of_week.to_numpy() == dow
            out[rows] = target[rows].join(m.rename("p"), on=KEYS)["p"].to_numpy()
        return out
    raise ValueError(kind)


val_table = pd.DataFrame({k: scores(df.loc[is_va, "sales"], baseline_predict(df, VALID_START, k)) for k in ["last", "mean14", "snaive7", "dow4"]}).T
display(val_table.round(4))

In [ ]:
# TODO 2（問答）
#
# (1) 哪一個 baseline 的 RMSLE 最低？「最近 4 週同一個星期幾」為什麼比「上週同一天」穩？
# (2) RMSLE、MAE、WAPE 的排名一樣嗎？為什麼可能不一樣？（提示：RMSLE 看比例、MAE 被大的序列主導）
# (3) 這些 baseline 都是「每條序列各自算」。一個全域的 LightGBM 可以利用哪些「跨序列」的資訊？
# 答：

---
## 3. 手寫：特徵工程與遞迴預測

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | lag 特徵（`groupby` + `shift`） | 3 |
| 3B | 滾動平均（先 shift 再 rolling） | 4 |
| 3C | 日期、發薪日、節日、促銷、油價 | 5 |
| 3D | 遞迴預測：預測值回填 | 6 |
| 3E | 洩漏檢查；對照 baseline | 7 |

`make_features(df, min_lag)` 一次把所有特徵加到 DataFrame 上：`min_lag = 16` → 直接法（所有 lag ≥ 16）；`min_lag = 1` → 遞迴法用的模型。

### 3A. lag 特徵

#### 符號表（3A–3C）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| `KEYS` | 序列的識別：`["store_nbr", "family"]` | |
| `g` | `df.groupby(KEYS)["sales"]` | |
| `lag_L` | $y_{i,t-L}$ | `g.shift(L)` |
| `rmean_L_w` | 從 lag $L$ 開始往前 $w$ 天的平均 | |
| `LAGS`、`WINDOWS` | 用哪些 lag、哪些窗口 | 直接法：`[16, 17, 18, 21, 28, 35, 364]`、`[7, 14, 28, 56]` |

**下標公式**：
- TODO 3：`for L in lags: out[f"lag_{L}"] = g.shift(L)`。
- TODO 4：`base = g.shift(min_lag)`；`out[f"rmean_{min_lag}_{w}"] = base.groupby([out[k] for k in KEYS]).transform(lambda x: x.rolling(w, min_periods=1).mean())`。

#### 最小例子：TODO 3、4

兩條序列 A、B，各 4 天，**依日期交錯**排列的話，`df["sales"].shift(1)` 會把 B 的值移到 A —— 所以要先依（序列, 日期）排好，再用 `groupby(...).shift`：

| 序列 | 日期 1..4 的 sales | 正確的 lag 1 | 正確的 rmean（shift 1，窗口 2） |
| --- | --- | --- | --- |
| A | 1, 2, 3, 4 | NaN, 1, 2, 3 | NaN, 1, 1.5, 2.5 |
| B | 10, 20, 30, 40 | NaN, 10, 20, 30 | NaN, 10, 15, 25 |

In [ ]:
# 最小例子：TODO 3、4 —— 錯的做法
mini = pd.DataFrame({"store_nbr": [1] * 8, "family": ["A"] * 4 + ["B"] * 4, "date": list(pd.date_range("2020-01-01", periods=4)) * 2,
                     "sales": [1.0, 2.0, 3.0, 4.0, 10.0, 20.0, 30.0, 40.0], "onpromotion": 0})
print("df['sales'].shift(1)（沒有 groupby）：", mini["sales"].shift(1).tolist(), "← B 的第一天拿到了 A 的最後一天（4.0）")
# 寫完 TODO 3、4、執行過下一格之後，取消下面的註解來對照（見上表）：
# print(add_lags(mini, [1])["lag_1"].tolist(), add_rolling(mini, 1, [2])["rmean_1_2"].tolist())

In [ ]:
def add_lags(df: pd.DataFrame, lags: list[int]) -> pd.DataFrame:
    """df 已依 KEYS + date 排序、每條序列的日期連續。回傳加了 lag_L 欄的副本。"""
    out = df.copy()
    g = out.groupby(KEYS, sort=False)["sales"]
    # TODO 3（程式）：每個 L 一欄 lag_L
    raise NotImplementedError("TODO 3：lag 特徵")
    return out


def add_rolling(df: pd.DataFrame, min_lag: int, windows: list[int]) -> pd.DataFrame:
    """從 lag min_lag 開始往前 w 天的平均：rmean_{min_lag}_{w}。"""
    out = df.copy()
    base = out.groupby(KEYS, sort=False)["sales"].shift(min_lag)
    grouper = [out[k] for k in KEYS]
    for w in windows:
        # TODO 4（程式）：base 在每條序列內部的滾動平均（min_periods=1）
        raise NotImplementedError("TODO 4：滾動平均")
    return out

### 3C. 日期、發薪日、節日、促銷、油價

| 特徵 | 在預測時知道嗎？ |
| --- | --- |
| 星期幾、幾號、月份、一年的第幾天 | ✅ 日曆 |
| 發薪日（15 號、月底）、月初 | ✅ |
| 全國性節日（`holidays_events`：`locale == "National"`、沒有被移到別天）| ✅ 事先公布（地震這種「事件」除外 —— Kaggle 的資料裡有，現實中預測不到） |
| `onpromotion` | ✅ Kaggle 的 test 有給（促銷是事先排好的） |
| 油價 | ⚠️ Kaggle 有給到 test 期間；現實中要用「預測當天之前」的值。本章用 16 天前的油價（`oil_lag16`），避免偷看 |
| 店、類別 | ✅ 當類別特徵（`category`） |

**下標公式**：
- TODO 5：`dow = date.dt.day_of_week`、`dom = date.dt.day`、`month`、`doy = date.dt.day_of_year`；`payday = (dom == 15) | date.dt.is_month_end`；`is_holiday = date.isin(全國節日的日期)`。

#### 最小例子：TODO 5

日期 2017-08-14、08-15、08-16、08-31 → `dow` $(0, 1, 2, 3)$；`payday` $(0, 1, 0, 1)$。

In [ ]:
def national_holidays(hol: pd.DataFrame | None) -> pd.DatetimeIndex:
    """全國性、沒有被調移的節日／事件日期（Kaggle 的 holidays_events 格式）。"""
    if hol is None:
        return pd.DatetimeIndex([])
    h = hol.assign(date=pd.to_datetime(hol["date"]))
    keep = (h["locale"] == "National") & (~h["transferred"].astype(bool)) & (h["type"] != "Work Day")
    return pd.DatetimeIndex(h.loc[keep, "date"].unique())


HOLIDAY_DATES = national_holidays(holidays_raw)


def add_calendar(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    d = out["date"]
    # TODO 5（程式）：dow、dom、month、doy、payday（0/1）、is_holiday（0/1）（見下標公式）
    raise NotImplementedError("TODO 5：日期特徵")
    return out


# 寫完 TODO 5 之後，取消下面的註解來對照（dow [0 1 2 3]、payday [0 1 0 1]）：
# print(add_calendar(pd.DataFrame({"date": pd.to_datetime(["2017-08-14", "2017-08-15", "2017-08-16", "2017-08-31"])}))[["dow", "payday"]].T)

In [ ]:
def add_oil(df: pd.DataFrame, lag: int = 16) -> pd.DataFrame:
    """lag 天前的油價（週末缺值用前一個值補）。"""
    if oil_raw is None:
        return df
    oil = oil_raw.assign(date=pd.to_datetime(oil_raw["date"])).set_index("date")["dcoilwtico"]
    oil = oil.reindex(pd.date_range(oil.index.min(), df["date"].max(), freq="D")).ffill().bfill()
    return df.assign(oil_lag16=df["date"].map(oil.shift(lag)))


def make_features(df: pd.DataFrame, min_lag: int) -> tuple[pd.DataFrame, list[str]]:
    """回傳 (加了特徵的 df, 特徵欄名)。min_lag = 16：直接法；1：遞迴法。"""
    lags = sorted(set([min_lag, min_lag + 1, min_lag + 2, min_lag + 5] + [L for L in (7, 14, 21, 28, 35, 364) if L >= min_lag]))
    out = add_lags(df, lags)
    out = add_rolling(out, min_lag, [7, 14, 28, 56])
    out = add_calendar(out)
    out = add_oil(out)
    out["store_nbr_cat"] = out["store_nbr"].astype("category")
    out["family_cat"] = out["family"].astype("category")
    feats = [c for c in out.columns if c.startswith(("lag_", "rmean_"))] + ["dow", "dom", "month", "doy", "payday", "is_holiday", "onpromotion", "store_nbr_cat", "family_cat"]
    if "oil_lag16" in out:
        feats.append("oil_lag16")
    return out, feats


t0 = time.perf_counter()
feat_df, FEATS = make_features(df, min_lag=H)
print(f"直接法特徵：{len(FEATS)} 個；{time.perf_counter() - t0:.0f} 秒")
print(FEATS)

### 3D. 遞迴預測

遞迴法的模型用 lag 1 起的特徵（`min_lag = 1`），只能預測「明天」。預測 16 天：

```
for 每一天 d（valid 的第 1 天到第 16 天）：
    1. 把「到 d 為止的資料」（d 之前已經用預測值填好的）重新算特徵
    2. 取出 d 那一天的列，用模型預測
    3. 把預測值寫回 d 那一天的 sales —— 第 d + 1 天的 lag 1 就會用到它
```

為了快，每一步只拿「最近 400 天」的資料重新算特徵（最長的 lag 是 364、滾動窗口 56）。

#### 符號表（3D）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| `work` | 最近 400 天 + 要預測的 16 天；預測的那幾天 sales 先設 NaN | DataFrame |
| `predict_fn` | 特徵 DataFrame → 預測的 sales（已經 expm1、clip 過） | 函式 |
| `day_rows` | `work` 中日期 = d 的列 | 布林 |

**下標公式**：
- TODO 6：`feats_d, _ = feature_fn(work)`；`p = predict_fn(feats_d[day_rows])`；`work.loc[day_rows, "sales"] = p`；`preds[d] = p`。

#### 最小例子：TODO 6

一條序列，歷史 sales = (1, 2, 3, 4, 5)；「模型」= lag_1 + 1。遞迴預測 3 天：$6, 7, 8$（每一天都用前一天的「預測值」當 lag 1）。

In [ ]:
def recursive_forecast(df: pd.DataFrame, start: pd.Timestamp, horizon: int, feature_fn, predict_fn, history_days: int = 400) -> np.ndarray:
    """遞迴法：回傳 df 在 [start, start + horizon) 的預測（順序同 df 的那些列）。feature_fn(work) → (加了特徵的 work, 欄名)。"""
    end = start + pd.Timedelta(days=horizon)
    work = df[(df["date"] >= start - pd.Timedelta(days=history_days)) & (df["date"] < end)].copy()
    future = work["date"] >= start
    work.loc[future, "sales"] = np.nan
    for d in pd.date_range(start, end - pd.Timedelta(days=1)):
        day_rows = (work["date"] == d).to_numpy()
        if not day_rows.any():
            continue  # 例如 12/25 沒有資料
        # TODO 6（程式）：重算特徵 → 取出 d 那天 → 預測 → 寫回 work 的 sales（見下標公式）
        raise NotImplementedError("TODO 6：遞迴預測")
    target = df[(df["date"] >= start) & (df["date"] < end)]
    return target.join(work.set_index(KEYS + ["date"])["sales"].rename("pred"), on=KEYS + ["date"])["pred"].to_numpy()


# 寫完 TODO 6 之後，取消下面的註解來對照（應為 [6. 7. 8.]）：
# toy = pd.DataFrame({"store_nbr": 1, "family": "A", "date": pd.date_range("2020-01-01", periods=8), "sales": [1.0, 2, 3, 4, 5, 0, 0, 0], "onpromotion": 0}); print(recursive_forecast(toy, pd.Timestamp("2020-01-06"), 3, lambda w: (add_lags(w, [1]), ["lag_1"]), lambda f: f["lag_1"].to_numpy() + 1, history_days=10))

### 3E. 洩漏檢查

1. **lag 特徵是不是真的「L 天前、同一條序列」**：隨機抽幾列，用 `(store, family, date − L)` 去原始資料查 sales，跟 `lag_L` 比。
2. **test 的特徵有沒有用到 test 期間的銷售**：把 test 期間的 sales 全部改成 NaN 再算一次特徵 —— 直接法的特徵應該**完全不變**（遞迴法的 lag 1 就會變）。

In [ ]:
lookup = df.set_index(KEYS + ["date"])["sales"]
sample = feat_df[feat_df["date"] >= VALID_START].sample(2000, random_state=0)
for L in [16, 28, 364]:
    expected = lookup.reindex(pd.MultiIndex.from_arrays([sample["store_nbr"], sample["family"], sample["date"] - pd.Timedelta(days=L)])).to_numpy()
    ok = np.allclose(expected, sample[f"lag_{L}"].to_numpy(), equal_nan=True)
    print(f"✅ lag_{L} = 同一條序列 {L} 天前的 sales？", ok)

df_masked = df.copy()
df_masked.loc[df_masked["date"] >= TEST_START, "sales"] = np.nan
feat_masked, _ = make_features(df_masked, min_lag=H)
num_feats = [f for f in FEATS if not f.endswith("_cat")]
same = np.allclose(feat_masked.loc[is_te, num_feats].to_numpy(dtype=float), feat_df.loc[is_te, num_feats].to_numpy(dtype=float), equal_nan=True)
print("✅ 把 test 的 sales 遮掉，直接法的 test 特徵完全不變？", same)
feat_rec_masked, _ = make_features(df_masked, min_lag=1)
feat_rec, _ = make_features(df, min_lag=1)
print("   遞迴法用的特徵（lag 1）呢？test 期間有幾成的 lag_1 變了：", f"{np.mean(~np.isclose(feat_rec_masked.loc[is_te, 'lag_1'], feat_rec.loc[is_te, 'lag_1'], equal_nan=True)):.1%}")

In [ ]:
# TODO 7（問答）TODO 3–6 與洩漏檢查
#
# (1) lag 的檢查通過，是因為第 1 節先把每條序列「補齊每一天」（12/25 補 0）。如果沒有補齊，「shift 364 列」跟「364 天前」還一樣嗎？
#     哪些列會出錯？（提示：中間跨過幾個 12/25，就差幾天）—— 這是時間序列特徵工程最常見的 bug 之一
# (2) 遮掉 test 的 sales 後，直接法的特徵完全不變 —— 為什麼這個檢查很重要？如果用了 lag_1 當特徵、在 test 上用真實的 lag_1 評估，成績會被高估還是低估？
# (3) 為什麼油價要用 16 天前的？節日表裡的「地震」事件（2016 年）在現實中能事先知道嗎？
# 答：

---
## 4. 套件：LightGBM

目標 $z = \log(1 + y)$（前置複習 1）；預測 $\hat y = \max(0, e^{\hat z} - 1)$。另外，**valid 開始前 56 天全部是 0 的序列直接預測 0**（那些店不賣那一類）。

| 參數 | 本章的值 | 第一部分第 12 章的意義 |
| --- | --- | --- |
| `n_estimators` | 3000（搭配 early stopping） | 樹的數量上限 |
| `learning_rate` | 0.05 | 每棵樹的縮小倍數 |
| `num_leaves` | 63 | 每棵樹的葉子數（leaf-wise 的容量） |
| `min_child_samples` | 50 | 葉子最少的樣本數 |
| `subsample`、`colsample_bytree` | 0.8 | bagging、特徵抽樣 |
| `early_stopping(100)` | valid 的 L2 連續 100 輪沒進步就停 | |

**實驗**：
1. 直接法（`min_lag = 16`）vs 遞迴法（`min_lag = 1` + TODO 6）在 valid 上；
2. **特徵消融**：只有日曆 → + lag → + 滾動平均 → + 促銷／節日／油價；
3. 特徵重要性（gain）。

In [ ]:
import lightgbm as lgb

LGB_PARAMS = dict(n_estimators=3000, learning_rate=0.05, num_leaves=63, min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=0)


def zero_series(df: pd.DataFrame, before: pd.Timestamp, days: int = 56) -> pd.Index:
    recent = df[(df["date"] < before) & (df["date"] >= before - pd.Timedelta(days=days))]
    s = recent.groupby(KEYS)["sales"].sum()
    return s[s == 0].index


def fit_lgb(feat_df: pd.DataFrame, feats: list[str], tr_mask, va_mask, params: dict = LGB_PARAMS, early_stopping: bool = True) -> lgb.LGBMRegressor:
    tr = feat_df[tr_mask].dropna(subset=["sales"])
    model = lgb.LGBMRegressor(**params)
    if early_stopping:
        va = feat_df[va_mask]
        model.fit(tr[feats], np.log1p(tr["sales"]), eval_set=[(va[feats], np.log1p(va["sales"]))], callbacks=[lgb.early_stopping(100, verbose=False)])
    else:
        model.fit(tr[feats], np.log1p(tr["sales"]))
    return model


def predict_sales(model: lgb.LGBMRegressor, X: pd.DataFrame, feats: list[str], zeros: pd.Index | None = None) -> np.ndarray:
    pred = np.clip(np.expm1(model.predict(X[feats], num_iteration=getattr(model, "best_iteration_", None) or None)), 0, None)
    if zeros is not None and len(zeros):
        pred[pd.MultiIndex.from_frame(X[KEYS]).isin(zeros)] = 0.0
    return pred


ZEROS_VA = zero_series(df, VALID_START)
t0 = time.perf_counter()
m_direct = fit_lgb(feat_df, FEATS, is_tr, is_va)
pred_va_direct = predict_sales(m_direct, feat_df[is_va], FEATS, ZEROS_VA)
print(f"直接法：best_iteration {m_direct.best_iteration_}；{time.perf_counter() - t0:.0f} 秒")
val_table.loc["LightGBM direct (lag ≥ 16)"] = scores(df.loc[is_va, "sales"], pred_va_direct)
display(val_table.round(4))

In [ ]:
# 實驗 1：遞迴法（min_lag = 1）
t0 = time.perf_counter()
feat_rec_df, FEATS_REC = make_features(df, min_lag=1)
m_rec = fit_lgb(feat_rec_df, FEATS_REC, is_tr, is_va)
pred_va_rec = recursive_forecast(df, VALID_START, H, lambda w: make_features(w, min_lag=1), lambda f: predict_sales(m_rec, f, FEATS_REC, ZEROS_VA))
print(f"遞迴法：best_iteration {m_rec.best_iteration_}；含 16 步遞迴共 {time.perf_counter() - t0:.0f} 秒")
val_table.loc["LightGBM recursive (lag ≥ 1)"] = scores(df.loc[is_va, "sales"], pred_va_rec)
pred_va_cheat = predict_sales(m_rec, feat_rec_df[is_va], FEATS_REC, ZEROS_VA)
val_table.loc["(leaky) recursive model with TRUE lag_1"] = scores(df.loc[is_va, "sales"], pred_va_cheat)
display(val_table.round(4))

day_idx = (df.loc[is_va, "date"] - VALID_START).dt.days.to_numpy()
by_day = pd.DataFrame({name: [scores(df.loc[is_va, "sales"].to_numpy()[day_idx == k], p[day_idx == k])["rmsle"] for k in range(H)]
                       for name, p in [("direct", pred_va_direct), ("recursive", pred_va_rec), ("leaky", pred_va_cheat)]}, index=np.arange(1, H + 1))
by_day.plot(marker="o", figsize=(8, 3.5), title="valid RMSLE by horizon day")
plt.show()

In [ ]:
# 實驗 2：特徵消融（直接法）；實驗 3：特徵重要性
groups = {
    "calendar + ids": ["dow", "dom", "month", "doy", "payday", "store_nbr_cat", "family_cat"],
    "+ lags": [f for f in FEATS if f.startswith("lag_")],
    "+ rolling means": [f for f in FEATS if f.startswith("rmean_")],
    "+ promo / holiday / oil": ["onpromotion", "is_holiday"] + (["oil_lag16"] if "oil_lag16" in FEATS else []),
}
abl_rows, used = [], []
for name, fs in groups.items():
    used = used + fs
    m_ = fit_lgb(feat_df, used, is_tr, is_va)
    abl_rows.append({"features": name, "# features": len(used), **scores(df.loc[is_va, "sales"], predict_sales(m_, feat_df[is_va], used, ZEROS_VA)), "best_iter": m_.best_iteration_})
display(pd.DataFrame(abl_rows).set_index("features").round(4))
imp = pd.Series(m_direct.booster_.feature_importance("gain"), index=FEATS).sort_values(ascending=False)
(imp / imp.sum()).head(15).plot.barh(figsize=(7, 4.5), title="feature importance (gain share)")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# TODO 8（問答）
#
# (1) 直接法 vs 遞迴法：valid RMSLE 各多少？依「預測第幾天」看，遞迴法的誤差在後面幾天有沒有變大？為什麼？
# (2) 「遞迴模型 + 真實的 lag_1」（leaky）跟真正的遞迴法差多少？leaky 的成績在現實中不可能得到 —— 如果有人回報這個數字，錯在哪？
#     差距很大代表什麼（lag_1 帶了很多「未來」的資訊）？差距很小又代表什麼？
# (3) 特徵消融：哪一組特徵進步最多？只有日曆 + 店／類別的模型能做到多少？
# (4) 特徵重要性：前幾名是什麼？store、family 的重要性高代表什麼？
# 答：

---
## 5. 調參

時間序列的驗證：**只用 valid（16 天）**挑設定。三組 LightGBM 設定 × 直接法的完整特徵；early stopping 決定樹的數量。之後用 **train + valid** 重新訓練（樹的數量 = best_iteration × 1.1，多一點資料就多幾棵）、預測 test。

In [ ]:
configs = {
    "num_leaves=31": dict(LGB_PARAMS, num_leaves=31),
    "num_leaves=63": LGB_PARAMS,
    "num_leaves=255, min_child=100": dict(LGB_PARAMS, num_leaves=255, min_child_samples=100),
}
tune_rows = {}
for name, params in configs.items():
    if name == "num_leaves=63":
        m_, p_ = m_direct, pred_va_direct
    else:
        m_ = fit_lgb(feat_df, FEATS, is_tr, is_va, params)
        p_ = predict_sales(m_, feat_df[is_va], FEATS, ZEROS_VA)
    tune_rows[name] = {**scores(df.loc[is_va, "sales"], p_), "best_iter": m_.best_iteration_}
tune = pd.DataFrame(tune_rows).T
display(tune.round(4))
best_cfg = tune["rmsle"].idxmin()
print("valid RMSLE 最低：", best_cfg)

In [ ]:
# TODO 9（問答）
#
# (1) 三組設定差多少？best_iteration 各多少？num_leaves 大的需要比較少還是比較多棵樹？
# (2) valid 只有 16 天 —— 用一個 16 天的窗口選設定可靠嗎？你會怎麼讓它更可靠？（提示：第 1 章的滾動起點，往前多取幾個 16 天）
# (3) 為什麼最後要用 train + valid 重新訓練？樹的數量為什麼要稍微多一點？
# 答：

---
## 6. 測試集評估

用選出的設定、在 train + valid 上重新訓練（直接法），預測 test 的 16 天，只評估一次。參照點：最好的 baseline（在 test 上用同樣的規則）、遞迴法（用第 4 節的模型，只在 train 上訓練）。成績寫進 `results/store_sales.csv`。

In [ ]:
from ml_dl_llm.results import compare, log_result

BENCH = "store_sales"
final_params = dict(configs[best_cfg], n_estimators=int(tune.loc[best_cfg, "best_iter"] * 1.1))
m_final = fit_lgb(feat_df, FEATS, (df["date"] >= TRAIN_START) & (df["date"] < TEST_START), None, final_params, early_stopping=False)
ZEROS_TE = zero_series(df, TEST_START)
pred_te = predict_sales(m_final, feat_df[is_te], FEATS, ZEROS_TE)
y_te = df.loc[is_te, "sales"].to_numpy()
best_base = val_table.loc[["last", "mean14", "snaive7", "dow4"], "rmsle"].idxmin()
test_rows = {
    f"baseline: {best_base}": scores(y_te, baseline_predict(df, TEST_START, best_base)),
    "LightGBM recursive (train only)": scores(y_te, recursive_forecast(df, TEST_START, H, lambda w: make_features(w, min_lag=1), lambda f: predict_sales(m_rec, f, FEATS_REC, ZEROS_TE))),
    f"LightGBM direct {best_cfg} (train+valid)": scores(y_te, pred_te),
}
test_table = pd.DataFrame(test_rows).T
display(test_table.round(4))
for name, m_ in test_rows.items():
    log_result(BENCH, name, m_, horizon=H, n_series=n_series, **({"num_leaves": final_params["num_leaves"], "n_estimators": final_params["n_estimators"]} if "direct" in name else {}))
compare(BENCH, "rmsle", higher_is_better=False)

---
## 7. 診斷

1. 每個類別的 RMSLE（哪些類別最難？）；
2. 誤差 vs 序列的大小（平均銷售額）；
3. 有促銷 vs 沒促銷的列。

In [ ]:
te = df[is_te].assign(pred=pred_te, sq=lambda d: (np.log1p(d["pred"]) - np.log1p(d["sales"])) ** 2)
by_fam = te.groupby("family").agg(rmsle=("sq", lambda s: np.sqrt(s.mean())), mean_sales=("sales", "mean")).sort_values("rmsle", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
by_fam["rmsle"].head(15).plot.barh(ax=axes[0], title="test RMSLE by family (worst 15)")
axes[0].invert_yaxis()
series_err = te.groupby(KEYS).agg(rmsle=("sq", lambda s: np.sqrt(s.mean())), mean_sales=("sales", "mean"))
axes[1].scatter(np.log1p(series_err["mean_sales"]), series_err["rmsle"], s=8, alpha=0.5)
axes[1].set_xlabel("log1p(mean test sales) of the series")
axes[1].set_ylabel("series RMSLE")
plt.show()
print("有促銷 vs 沒促銷的 RMSLE：", te.groupby(te["onpromotion"] > 0)["sq"].apply(lambda s: float(np.sqrt(s.mean()))).round(4).to_dict())
print("全部是 0 而預測 0 的序列數：", len(ZEROS_TE), "；test 期間實際上有賣的：", int(te[pd.MultiIndex.from_frame(te[KEYS]).isin(ZEROS_TE)]["sales"].gt(0).sum()), "列")

In [ ]:
# TODO 10（問答，本章總結）
#
# (1) 測試集：LightGBM（直接法）比最好的 baseline 好多少？比遞迴法呢？跟 valid 上的排名一致嗎？
# (2) 哪些類別最難？是銷售很小（很多 0、log 尺度上的跳動大）還是很大的？RMSLE 為什麼對小的序列特別嚴格？
# (3) 「全部是 0 就預測 0」的規則：有沒有序列在 test 期間突然開始賣？這個規則什麼時候會出錯？
# (4) 跟第 1～7 章比：全域的 LightGBM 適合什麼樣的問題（很多序列、有外生變數、資料量大）？什麼時候統計模型比較好（一條短序列、需要預測區間）？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Store Sales（test = 最後 16 天）

| 方法 | 特徵 | valid RMSLE | test RMSLE | test WAPE |
| --- | --- | --- | --- | --- |
| 最好的 baseline | — | | | |
| LightGBM 直接法 | lag ≥ 16、滾動、日曆、促銷、節日、油價 | | | |
| LightGBM 遞迴法 | lag ≥ 1（預測值回填） | | | |
| （洩漏）遞迴模型 + 真實 lag 1 | — | | — | — |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ 直接法的變體：每個 horizon 一個模型（16 個模型）vs 一個模型 + 「第幾天」特徵
- ☐ 階層式預測（店 → 城市 → 全國）的一致性調整（reconciliation）
- ☐ Tweedie／Poisson 損失：直接處理很多 0 的非負目標

**延伸**
- ☐ `mlforecast`（Nixtla）：自動化 lag／滾動特徵、多個模型、交叉驗證
- ☐ 目標編碼、序列的統計特徵（平均、變異、零的比例）
- ☐ 用滾動起點（多個 16 天的窗口）做更穩的驗證
- ☐ M5 競賽（Walmart）：同一套方法的大型版本